# 08.4 - Encoder-Decoder Architecture

**Phase:** 08 - Transformers

**Status:** VERIFIED

---

## 1. What Are We Solving?

Some tasks change structure: a source sequence of one length becomes a target sequence of another length (translation, summarization, reversing). The original transformer design pairs an **encoder** that reads the whole source with a **decoder** that writes the target one token at a time.

## 2. Why Does This Matter?

This is the "Attention Is All You Need" (2017) architecture - the ancestor of T5 and BART and still the right choice for tasks with genuinely different input/output structures. Cross-attention and teacher forcing are ideas you will use everywhere.

## 3. Prerequisites

- Transformer block (08.1), self-attention + causal mask (08.2), positional encoding (08.3)

## 4. Learning Objectives

By the end of this unit, you should:
- Connect an encoder stack to a decoder stack with cross-attention
- Use teacher forcing during training
- Decode greedily at inference
- Explain why the decoder needs a causal mask and the encoder does not

## 5. Mental Model

```text
Input -> Encoder stack -> memory
                              \
                               Decoder stack <- previously written tokens
                              /
                        Output token

Encoder: reads everything (bidirectional).
Decoder: writes one token at a time (causal).
Cross-attention (in decoder): Q from decoder, K & V from encoder memory.
```


## 6. Setup


In [1]:
import matplotlib
matplotlib.use('Agg')
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
print('torch', torch.__version__)


torch 2.13.0+cpu


## 7. Attention, Encoder Block, Decoder Block

We write every piece by hand so we control the masks exactly. The decoder block has THREE sub-layers: masked self-attention, **cross-attention** (queries from the decoder, keys/values from the encoder memory), and a feed-forward network.


In [2]:
def causal_mask(n):
    return torch.tril(torch.ones(n, n, dtype=torch.bool))

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.h = n_heads
        self.dk = d_model // n_heads
        self.wq, self.wk, self.wv, self.wo = (nn.Linear(d_model, d_model)
                                              for _ in range(4))

    def forward(self, x, memory=None, mask=None):
        B, T, _ = x.shape
        if memory is None:
            memory = x
        q = self.wq(x).view(B, T, self.h, self.dk).transpose(1, 2)
        k = self.wk(memory).view(B, memory.size(1), self.h, self.dk).transpose(1, 2)
        v = self.wv(memory).view(B, memory.size(1), self.h, self.dk).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)
        if mask is not None:
            scores = scores.masked_fill(~mask[None, None], float('-inf'))
        w = F.softmax(scores, dim=-1)
        o = (w @ v).transpose(1, 2).reshape(B, T, -1)
        return self.wo(o), w

class EncoderBlock(nn.Module):
    '''self-attention + FFN + residuals + LayerNorms (no mask)'''
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                               nn.Linear(4 * d_model, d_model))
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        a, _ = self.attn(x)
        x = self.norm1(x + a)
        return self.norm2(x + self.ff(x))

class DecoderBlock(nn.Module):
    '''masked self-attn + CROSS-attn + FFN'''
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.sa = MultiHeadAttention(d_model, n_heads)
        self.ca = MultiHeadAttention(d_model, n_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, 4 * d_model), nn.GELU(),
                               nn.Linear(4 * d_model, d_model))

    def forward(self, x, memory, mask):
        a, _ = self.sa(x, mask=mask)          # causal self-attention
        x = self.norm1(x + a)
        c, w_cross = self.ca(x, memory)        # Q=x, K/V=memory (encoder)
        x = self.norm2(x + c)
        return self.norm3(x + self.ff(x)), w_cross

print('EncoderBlock and DecoderBlock defined - the decoder has 3 sub-layers.')


EncoderBlock and DecoderBlock defined - the decoder has 3 sub-layers.


## 8. Assemble the Encoder-Decoder Transformer


In [3]:
class EncoderDecoder(nn.Module):
    def __init__(self, vocab_size, d_model=24, n_heads=4, n_layers=2, max_len=16):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Parameter(torch.randn(1, max_len, d_model) * 0.02)
        self.encoder = nn.ModuleList([EncoderBlock(d_model, n_heads) for _ in range(n_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(d_model, n_heads) for _ in range(n_layers)])
        self.out = nn.Linear(d_model, vocab_size)

    def forward(self, src, tgt, mask=None):
        se = self.emb(src) + self.pos[:, :src.size(1)]
        for b in self.encoder:
            se = b(se)
        memory = se
        te = self.emb(tgt) + self.pos[:, :tgt.size(1)]
        for b in self.decoder:
            te, _ = b(te, memory, mask)
        return self.out(te), memory

model = EncoderDecoder(vocab_size=20)
src = torch.randint(1, 20, (2, 8))
tgt = torch.randint(1, 20, (2, 6))
logits, memory = model(src, tgt, mask=causal_mask(6))
print('src    :', tuple(src.shape))
print('memory :', tuple(memory.shape))
print('logits :', tuple(logits.shape), '-> vocab logits for every target position')
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('trainable parameters:', f'{n_train:,}')


src    : (2, 8)
memory : (2, 8, 24)
logits : (2, 6, 20) -> vocab logits for every target position
trainable parameters: 35,156


## 9. Cross-Attention: Decoder Looks at Encoder Memory

Prove the decoder consumes encoder memory: scramble the memory and watch decoder logits move.


In [4]:
with torch.no_grad():
    logits_a, _ = model(src, tgt, mask=causal_mask(6))
# rebuild memory with token embeddings only (no attention) - a useless 'encoder'
mem_b = model.emb(src) + model.pos[:, :src.size(1)]
te = model.emb(tgt) + model.pos[:, :tgt.size(1)]
for b in model.decoder:
    te, _ = b(te, mem_b, causal_mask(6))
logits_b = model.out(te)
diff = (logits_a - logits_b).abs().mean().item()
print(f'Mean |logits change| after removing encoder attention: {diff:.3f}')
print('If cross-attention did not read encoder memory this would be ~0. It is not.')


Mean |logits change| after removing encoder attention: 0.063
If cross-attention did not read encoder memory this would be ~0. It is not.


## 10. Task: Reverse a Sequence (a tiny seq2seq)

Train on '3 0 7 1 2 5' -> '5 2 1 7 0 3'. Teacher forcing: decoder input at inference time is BOS + *gold* previous tokens.


In [5]:
V, L, BOS = 8, 6, 0   # tokens 1..7, sequence length 6, BOS ids 0
vocab = V + 1

def gen_batch(batch):
    src = torch.randint(1, V + 1, (batch, L))
    tgt = torch.flip(src, dims=[1])
    dec_in = torch.cat([torch.full((batch, 1), BOS), tgt[:, :-1]], dim=1)
    return src, dec_in, tgt

src, dec_in, tgt = gen_batch(4)
print('src   :', src[0].tolist())
print('dec_in:', dec_in[0].tolist())
print('tgt   :', tgt[0].tolist())


src   : [5, 4, 8, 3, 1, 5]
dec_in: [0, 5, 1, 3, 8, 4]
tgt   : [5, 1, 3, 8, 4, 5]


## 11. Train with Teacher Forcing


In [6]:
model = EncoderDecoder(vocab_size=vocab, d_model=24, n_heads=4, n_layers=2, max_len=16)
opt = torch.optim.Adam(model.parameters(), lr=3e-3)
loss_fn = nn.CrossEntropyLoss()

for step in range(1, 601):
    src, dec_in, tgt = gen_batch(32)
    opt.zero_grad()
    logits, _ = model(src, dec_in, mask=causal_mask(L))
    loss = loss_fn(logits.reshape(-1, vocab), tgt.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 200 == 0:
        print(f'step {step:3d}: loss={loss.item():.3f}')
print('\nTeacher-forced loss collapses - the decoder learns reversed order.')


step 200: loss=0.142


step 400: loss=0.028


step 600: loss=0.001

Teacher-forced loss collapses - the decoder learns reversed order.


## 12. Inference: Greedy Decoding (No Teacher)

At inference we feed the model's *own* previous prediction, one token at a time.


In [7]:
def greedy_decode(model, src):
    batch = src.size(0)
    tgt = torch.full((batch, 1), BOS)
    for _ in range(L):
        logits, _ = model(src, tgt, mask=causal_mask(tgt.size(1)))
        nxt = logits[:, -1].argmax(-1).unsqueeze(1)
        tgt = torch.cat([tgt, nxt], dim=1)
    return tgt[:, 1:]

model.eval()
with torch.no_grad():
    src_test, _, tgt_test = gen_batch(512)
    preds = greedy_decode(model, src_test)
    acc = (preds == tgt_test).all(dim=1).float().mean().item()
print(f'Exact-sequence greedy-decoding accuracy: {acc*100:.0f}%')
i = 5
print(f'src:   {src_test[5].tolist()}')
print(f'out:   {preds[5].tolist()}')
print(f'tgt:   {tgt_test[5].tolist()}')


Exact-sequence greedy-decoding accuracy: 100%
src:   [4, 1, 2, 2, 4, 7]
out:   [7, 4, 2, 2, 1, 4]
tgt:   [7, 4, 2, 2, 1, 4]


## 13. Failure Case: Removing the Causal Mask at Inference

The model was trained under a causal mask, so it expects position i to see only 0..i. Remove the mask and position i suddenly blends in the *future* gold tokens - the exact information that should never be visible. The decoder's behavior changes dramatically and is no longer a valid autoregressive predictor.


In [8]:
src, dec_in, tgt = gen_batch(64)
with torch.no_grad():
    l_masked, _ = model(src, dec_in, mask=causal_mask(L))
    l_unmasked, _ = model(src, dec_in, mask=None)   # broken: sees future tokens

def top1_prob(logits, tgt):
    p = F.softmax(logits, dim=-1)
    return p.gather(-1, tgt.unsqueeze(-1)).squeeze(-1).mean().item()

print(f'P(gold token) with causal mask:  {top1_prob(l_masked, tgt):.3f}')
print(f'P(gold token) WITHOUT mask:      {top1_prob(l_unmasked, tgt):.3f}')
print(f'Mean |logits change| when mask removed: '
      f'{(l_masked - l_unmasked).abs().mean().item():.3f}')
print('\nA model trained WITHOUT a mask gets trivially low training loss (it cheats),')
print('then collapses at inference when future tokens no longer exist.')


P(gold token) with causal mask:  0.999


P(gold token) WITHOUT mask:      0.575
Mean |logits change| when mask removed: 1.677

A model trained WITHOUT a mask gets trivially low training loss (it cheats),
then collapses at inference when future tokens no longer exist.


## 14. Debugging

| Symptom | Possible Cause | Verify | Fix |
|---|---|---|---|
| Decoder repeats tokens | Mask too permissive | Check mask is lower-triangular | Apply causal mask |
| Training loss flat | Cross-attention disconnected | Perturb memory, watch logits | Wire memory into decoder |
| Train good / inference bad | Teacher-forcing exposure bias | Compare forced vs free-run | Scheduled sampling |
| Output length fixed | No EOS handling | Check if model learns EOS | Add EOS, use beam search |

## 15. Real-World Considerations

- Share the embedding matrix between encoder and decoder when vocabularies match (fewer parameters).
- T5 and BART are production encoder-decoder transformers; GPT-style models are decoder-only with no cross-attention.
- Teacher forcing optimizes next-token accuracy, not sequence accuracy - evaluate with greedy decode or beam search.

## 16. Common Mistakes

- Forgetting the causal mask (cheating).
- Bidirectional attention in the decoder (breaks autoregression).
- Confusing cross-attention (Q=decoder, K/V=encoder) with self-attention.
- Evaluating with teacher forcing instead of free-run decoding.

## 17. When NOT to Use

- Simple classification: encoder-only (BERT) is cheaper.
- Open-ended generation: decoder-only (GPT) is simpler.
- Input and output have the same structure: a decoder-only model may suffice.

## 18. Challenge: Scheduled Sampling

Occasionally feed the model's own predictions instead of gold tokens during training. This should keep (or improve) the free-run decoding accuracy.


In [9]:
model.train()
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
loss_fn = nn.CrossEntropyLoss()
e = 0.3  # 30% of batches sampled from the model itself
for step in range(200):
    src, dec_in, tgt = gen_batch(32)
    if torch.rand(1).item() < e:
        with torch.no_grad():
            pred_full = greedy_decode(model, src)
        dec_in = torch.cat([torch.full((len(src), 1), BOS), pred_full[:, :-1]], dim=1)
    opt.zero_grad()
    logits, _ = model(src, dec_in, mask=causal_mask(L))
    loss = loss_fn(logits.reshape(-1, vocab), tgt.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

model.eval()
with torch.no_grad():
    src_t, _, tgt_t = gen_batch(512)
    acc = (greedy_decode(model, src_t) == tgt_t).all(1).float().mean().item()
print(f'Greedy-decoding accuracy after scheduled sampling: {acc*100:.0f}%')
print('Mixing gold and predicted inputs reduces exposure bias.')


Greedy-decoding accuracy after scheduled sampling: 99%
Mixing gold and predicted inputs reduces exposure bias.


## 19. Closed-Book Recall

1. In cross-attention, where do Q, K, V come from?
2. What is teacher forcing and what problem does it cause?
3. Why does the decoder need a causal mask but the encoder does not?
4. How do you generate a sequence at inference time?

## 20. Teach-Back Questions

Explain to another person:

- The encoder-reads / decoder-writes mental model with cross-attention.
- Why the unmasked decoder cheats during training.
- The reverse-sequence task and greedy decoding.

## 21. Summary

You built an encoder-decoder from scratch with explicit cross-attention, proved the decoder reads encoder memory, trained the model to reverse sequences with teacher forcing, decoded greedily at inference, reproduced the missing-mask failure, and applied scheduled sampling.

## 22. Further Experiment

- Train to reverse length-12 sequences and watch accuracy drop.
- Implement beam search (width 3) and compare exact-match accuracy.
- Add EOS and train open-ended generation instead of fixed length.

## 23. Verification Status

```
STATUS: VERIFIED
EXECUTION: PASS
DEPENDENCIES: numpy, torch
OUTPUTS: PASS
LAST VERIFIED: 2026-08-29
```
